# Chapter 4: Training a Neural Network and Computational Thinking

In [2]:
import torch
import matplotlib.pyplot as plt
import numpy as np
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
# At the top of your notebook, add:
np.set_printoptions(suppress=True, precision=8)
torch.set_printoptions(sci_mode=False, precision=8)

cuda


# 1. Understanding Modules and Layers in PyTorch

## Parameter Registration

In [ ]:
class BrokenLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        # Just a regular tensor
        self.weight = torch.randn(out_dim, in_dim)
        self.bias = torch.zeros(out_dim)
        
        # Correct way.
        # self.weight = nn.Parameter(torch.randn(out_dim, in_dim))
        # self.bias = nn.Parameter(torch.zeros(out_dim))
        
    def forward(self, x):
        return x @ self.weight.T + self.bias

layer = BrokenLayer(10, 5)
print(f"Number of parameters: {sum(p.numel() for p in layer.parameters())}")  # 0
print(f"Requires grad on weight? {layer.weight.requires_grad}")  # False

# The optimizer will have nothing to optimize!
optimizer = torch.optim.SGD(layer.parameters(), lr=0.01)
print(f"Optimizer param groups: {len(optimizer.param_groups[0]['params'])}")  # 0

In [ ]:
class Linear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        # These are automatically registered as parameters
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.bias = nn.Parameter(torch.Tensor(out_features))
        
        # Alternative manual registration:
        # self.register_parameter('weight', nn.Parameter(...))

## Module Hierarchy: Parameters

In [ ]:
class Network(nn.Module):
    def __init__(self):
        super().__init__()
        # Child modules are automatically registered
        self.layer1 = nn.Linear(10, 20)
        self.layer2 = nn.Linear(20, 5)
        
    def forward(self, x):
        return self.layer2(self.layer1(x))
        
# Accessing the hierarchy:
model = Network()
print(list(model.children()))  # [layer1, layer2]


In [ ]:
print(list(model.parameters()))  # All parameters from both layers

In [ ]:
class CustomLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_features, out_features))
        self.bias = nn.Parameter(torch.zeros(out_features))
    
    def forward(self, x):
        return x @ self.weight + self.bias

## An Example of a Complex Network

In [ ]:
class ComplexNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Submodule 1: Feature extractor
        self.feature_extractor = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Submodule 2: Processor
        self.processor = nn.ModuleList([
            nn.Linear(256, 128),
            nn.Linear(128, 64)
        ])
        
        # Submodule 3: Output head
        self.classifier = nn.Linear(64, 10)
        
    def forward(self, x):
        x = self.feature_extractor(x)
        for layer in self.processor:
            x = layer(x)
        return self.classifier(x)

# Inspecting the hierarchy
model = ComplexNetwork()
print(f"Number of parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Module structure:\n{model}")

# Accessing specific parts
print(f"Feature extractor parameters: {list(model.feature_extractor.parameters())}")

## A Layer with a Conditional Parameter

In [ ]:
class ConditionalLayer(nn.Module):
    """Layer that sometimes has bias, sometimes doesn't"""
    def __init__(self, in_dim, out_dim, use_bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_dim, in_dim))
        self.use_bias = use_bias
        
        if use_bias:
            self.bias = nn.Parameter(torch.zeros(out_dim))
        else:
            # Important: Register bias as None
            self.register_parameter('bias', None)
            
    def forward(self, x):
        result = x @ self.weight.T
        if self.bias is not None:
            result = result + self.bias
        return result

# Both versions work correctly with optimizers
layer1 = ConditionalLayer(10, 5, use_bias=True)
layer2 = ConditionalLayer(10, 5, use_bias=False)
print(f"Layer1 params: {len(list(layer1.parameters()))}")  # 2
print(f"Layer2 params: {len(list(layer2.parameters()))}")  # 1

## Hooks

### Forward Hook

In [6]:
import torch
import torch.nn as nn

# Define a simple model
model = nn.Linear(5, 3)

# Define a hook function
def print_output(module, input, output):
    print(module)
    print(f"Iput: {input}")
    print(f"Output: {output}")

# Register the hook
handle = model.register_forward_hook(print_output)

# Run the model
x = torch.randn(1, 5)
print(x)
output = model(x)

# Remove the hook
handle.remove()

tensor([[ 1.47666156,  0.84247339, -1.61149466,  0.28967109, -0.62255901]])
Linear(in_features=5, out_features=3, bias=True)
Iput: (tensor([[ 1.47666156,  0.84247339, -1.61149466,  0.28967109, -0.62255901]]),)
Output: tensor([[ 0.04884279,  0.13909318, -0.44066066]], grad_fn=<AddmmBackward0>)


### Backward Hook

In [12]:
import torch
import torch.nn as nn

model = nn.Linear(5, 3)
model.weight.data = torch.tensor([
    [0.1, 0.2, 0.3, 0.4, 0.5],
    [0.6, 0.7, 0.8, 0.9, 1.0],
    [1.1, 1.2, 1.3, 1.4, 1.5]
])
model.bias.data = torch.tensor([0.1, 0.2, 0.3])

def print_gradients(module, grad_input, grad_output):
    print(f"Grad Input: {grad_input}")
    print(f"Grad Output: {grad_output}")

handle = model.register_backward_hook(print_gradients)

x = torch.tensor([[1., 2., 3., 4., 5.]])
print(x)
output = model(x)
loss = output.sum()
loss.backward()

handle.remove()

tensor([[1., 2., 3., 4., 5.]])
Grad Input: (tensor([1., 1., 1.]), None, tensor([[1., 1., 1.],
        [2., 2., 2.],
        [3., 3., 3.],
        [4., 4., 4.],
        [5., 5., 5.]]))
Grad Output: (tensor([[1., 1., 1.]]),)


Grad Input Tuple: (1): initial gradient input (2) Gradient with respect to bias (not provided by Hook), (3) Initial Gradient with respect to W (4) Out gradient with respect to W.

In [13]:
print("Weight gradient:", model.weight.grad)
print("Bias gradient:", model.bias.grad)

Weight gradient: tensor([[1., 2., 3., 4., 5.],
        [1., 2., 3., 4., 5.],
        [1., 2., 3., 4., 5.]])
Bias gradient: tensor([1., 1., 1.])


In [16]:
import torch
import torch.nn as nn

class HookedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 5)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(5, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# 1. Define the Forward Hook (peeking at activations)
def forward_hook_fn(module, input, output):
    print(f"--- Forward Hook: {module.__class__.__name__} ---")
    print(f"Input Shape: {input[0].shape}")
    print(f"Output Shape: {output.shape}\n")

# 2. Define the Backward Hook (peeking at gradients)
def backward_hook_fn(module, grad_input, grad_output):
    print(f"--- Backward Hook: {module.__class__.__name__} ---")
    # grad_output is the gradient of the loss w.r.t the output of this layer
    print(f"Gradient Output Norm: {grad_output[0].norm().item():.4f}\n")

# --- Execution ---
model = HookedModel()

# Register hooks on the ReLU layer specifically
handle_f = model.relu.register_forward_hook(forward_hook_fn)
handle_b = model.relu.register_full_backward_hook(backward_hook_fn)

# Dummy Input
data = torch.randn(1, 10)
target = torch.randn(1, 2)

# Forward Pass
output = model(data)

# Backward Pass
loss = torch.nn.functional.mse_loss(output, target)
loss.backward()

# Clean up (it's good practice to remove hooks after debugging)
handle_f.remove()
handle_b.remove()

--- Forward Hook: ReLU ---
Input Shape: torch.Size([1, 5])
Output Shape: torch.Size([1, 5])

--- Backward Hook: ReLU ---
Gradient Output Norm: 1.5828



In [ ]:
# df = pl.read_csv("https://raw.githubusercontent.com/rahulbhadani/CPE487587_SP26/refs/heads/master/Data/ResourceAssessmentSummaryData032011.csv",
#                 schema_overrides={
#         "Design Head (feet) ": pl.Utf8,
#         "Design Flow (cfs)": pl.Utf8,
#         "Installed Capacity (kW)": pl.Utf8,
#         "Annual Production (MWh)": pl.Utf8,
#         "Plant Factor": pl.Utf8,
#         "Total Construction Cost (1,000 $)": pl.Utf8,
#         "Annual O&M Cost (1,000 $)": pl.Utf8,
#         "Cost per Installed Capacity ($/kW)": pl.Utf8,
#         "IRR with Green Incentives": pl.Utf8,
#     }
# )

# # Remove quotes, commas, and dollar signs, then convert to float
# df = df.with_columns(
#     pl.col([
#         "Design Head (feet) ",
#         "Design Flow (cfs)",
#         "Installed Capacity (kW)",
#         "Annual Production (MWh)",
#         "Plant Factor",
#         "Total Construction Cost (1,000 $)",
#         "Annual O&M Cost (1,000 $)",
#         "Cost per Installed Capacity ($/kW)",
#         "IRR with Green Incentives",
#     ])
#     .str.replace_all(r'["\$,]', '')  # remove quotes, $, and commas
#     .str.replace_all(r'[<>]', '')     # remove < and >
#     .cast(pl.Float64)
# )

In [ ]:
# df = pl.read_csv("https://raw.githubusercontent.com/rahulbhadani/CPE487587_SP26/refs/heads/master/Data/ResourceAssessmentSummaryData032011.csv",
#                 schema_overrides={
#         "Design Head (feet) ": pl.Utf8,
#         "Design Flow (cfs)": pl.Utf8,
#         "Installed Capacity (kW)": pl.Utf8,
#         "Annual Production (MWh)": pl.Utf8,
#         "Plant Factor": pl.Utf8,
#         "Total Construction Cost (1,000 $)": pl.Utf8,
#         "Annual O&M Cost (1,000 $)": pl.Utf8,
#         "Cost per Installed Capacity ($/kW)": pl.Utf8,
#         "IRR with Green Incentives": pl.Utf8,
#     }
# )

# # Remove quotes, commas, and dollar signs, then convert to float
# df = df.with_columns(
#     pl.col([
#         "Design Head (feet) ",
#         "Design Flow (cfs)",
#         "Installed Capacity (kW)",
#         "Annual Production (MWh)",
#         "Plant Factor",
#         "Total Construction Cost (1,000 $)",
#         "Annual O&M Cost (1,000 $)",
#         "Cost per Installed Capacity ($/kW)",
#         "IRR with Green Incentives",
#     ])
#     .str.replace_all(r'["\$,]', '')  # remove quotes, $, and commas
#     .str.replace_all(r'[<>]', '')     # remove < and >
#     .cast(pl.Float64)
# )

# 2. Neural Network Training

In [18]:
df = pl.read_csv("https://raw.githubusercontent.com/rahulbhadani/CPE486586_FA25/refs/heads/main/Data/Hydropower.csv")
df

FacilityName,BCR,AnnualProduction,ConstructionCost,DesignHead,Y,X1,X2,X3
str,f64,i64,f64,i64,f64,f64,f64,f64
"""Anchor Dam """,0.02,126,5656.5,60,-3.912023,4.836282,8.640561,4.094345
"""Angostura Dam """,0.9,3218,3179.2,119,-0.105361,8.076515,8.064385,4.779123
"""Barretts Diversion Dam """,0.35,546,1391.4,15,-1.049822,6.302619,7.238066,2.70805
"""Belle Fourche Dam """,0.49,1319,2376.3,50,-0.71335,7.184629,7.7733,3.912023
"""Bonny Dam """,0.15,238,1476.8,70,-1.89712,5.472271,7.297633,4.248495
…,…,…,…,…,…,…,…,…
"""Upper Diamond Fork Flow Contro…",2.36,52161,22058.5,547,0.858662,10.86209,10.001453,6.304449
"""Upper Stillwater Dam """,0.32,1904,6064.5,161,-1.139434,7.551712,8.710207,5.081404
"""Vega Dam """,0.51,1702,3012.5,90,-0.673345,7.439559,8.010526,4.49981


We want to estimat annual production based BCR, ConstructionCost and DesignHead

In [19]:
class SimpleNN(nn.Module):
    def __init__(self, in_features):
        super(SimpleNN, self).__init__()
        self.in_features = in_features
        self.fc1 = nn.Linear(self.in_features, 64)
        self.fc2 = nn.Linear(64, 128)
        self.fc3 = nn.Linear(128, 16)
        self.fc4 = nn.Linear(16, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [20]:
# Convert Polars DataFrame to numpy arrays
X = df.drop(['FacilityName', 'Y', 'X1', 'X2', 'X3', 'AnnualProduction']).to_numpy() 
y = df['AnnualProduction'].to_numpy()     

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [23]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32, device=device)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32, device=device)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32, device=device)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32, device=device)

In [25]:
model = SimpleNN(in_features = X_train_tensor.shape[1]).to(device)


In [ ]:
# Define the loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001)
# Train the model
epochs = 100000

losses = torch.zeros(epochs, device=device)

for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    
    losses[epoch] = loss
    optimizer.step()
    if (epoch + 1) % 1000 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item()}')
        print(f'GPU Memory: {torch.cuda.memory_allocated() / 1024**2:.5f} MB')

## Batch Training

In [17]:
from torch.utils.data import TensorDataset, DataLoader

In [27]:
# Create TensorDataset and DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Set batch size
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [29]:
# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleNN(in_features=X_train_tensor.shape[1]).to(device)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001) 

# Training with batching
epochs = 100000
train_losses = []
val_losses = []

for epoch in range(epochs):
    # Training phase
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(test_loader)
    val_losses.append(avg_val_loss)
    
    if (epoch + 1) % 1000 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}')
        if torch.cuda.is_available():
            print(f'GPU Memory: {torch.cuda.memory_allocated() / 1024**2:.2f} MB')

# Evaluate on full test set
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor.to(device))
    test_loss = criterion(test_outputs, y_test_tensor.to(device))
    print(f'\nFinal Test Loss: {test_loss.item():.6f}')

Epoch 1000/100000, Train Loss: 153421676.800000, Val Loss: 479495856.000000
GPU Memory: 17.43 MB
Epoch 2000/100000, Train Loss: 152254202.000000, Val Loss: 478980864.000000
GPU Memory: 17.43 MB
Epoch 3000/100000, Train Loss: 153161674.400000, Val Loss: 477706320.000000
GPU Memory: 17.43 MB
Epoch 4000/100000, Train Loss: 151205769.600000, Val Loss: 475288960.000000
GPU Memory: 17.43 MB
Epoch 5000/100000, Train Loss: 148127569.600000, Val Loss: 471325056.000000
GPU Memory: 17.43 MB
Epoch 6000/100000, Train Loss: 145599874.800000, Val Loss: 465570096.000000
GPU Memory: 17.43 MB
Epoch 7000/100000, Train Loss: 143332967.600000, Val Loss: 457987488.000000
GPU Memory: 17.43 MB
Epoch 8000/100000, Train Loss: 137345864.800000, Val Loss: 449051584.000000
GPU Memory: 17.43 MB
Epoch 9000/100000, Train Loss: 133289412.400000, Val Loss: 439695376.000000
GPU Memory: 17.43 MB
Epoch 10000/100000, Train Loss: 152855357.800000, Val Loss: 431329552.000000
GPU Memory: 17.43 MB
Epoch 11000/100000, Train Los

KeyboardInterrupt: 

## Batch Normalization

In [ ]:
class SimpleNNWithBN(nn.Module):
    def __init__(self, in_features):
        super(SimpleNNWithBN, self).__init__()
        self.in_features = in_features
        
        # Layers with Batch Normalization
        self.fc1 = nn.Linear(self.in_features, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(64, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc3 = nn.Linear(128, 16)
        self.bn3 = nn.BatchNorm1d(16)
        self.fc4 = nn.Linear(16, 1)  # Output layer - no BN
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2) 

    def forward(self, x):
        # Layer 1: Linear -> BatchNorm -> Activation
        x = self.fc1(x)
        x = self.bn1(x)  # BatchNorm before activation
        x = self.relu(x)
        x = self.dropout(x) 
        
        # Layer 2
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        # Layer 3
        x = self.fc3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        # Output layer (no BN, no activation for regression)
        x = self.fc4(x)
        return x

In [ ]:
model = SimpleNNWithBN(in_features=X_train_tensor.shape[1]).to(device)

# Training parameters
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # Added weight decay
epochs = 200
batch_size = 32